# Client retention: SQL walkthrough
An executed walkthrough of the supplied SQLite analysis. Read the case study and metric dictionary alongside this notebook.
AI assisted implementation. Reproduce and understand each result before describing personal ownership.

## Connect to the supplied database
Run the notebook from this project folder.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
assert Path('analysis.sqlite').exists()
db = sqlite3.connect('analysis.sqlite')
print(pd.read_sql_query('SELECT cohort, COUNT(*) AS clients FROM features GROUP BY cohort', db).to_string(index=False))

  cohort  clients
 Holdout      406
Training      794
Upcoming      200


## Renewal by tier
Explain the denominator before comparing percentages.

In [2]:
print(pd.read_sql_query('SELECT tier, contracts, renewed_contracts, ROUND(100.0 * renewal_rate, 1) AS renewal_pct FROM renewal_by_tier', db).to_string(index=False))

        tier  contracts  renewed_contracts  renewal_pct
  Enterprise        172              128.0         74.4
   Essential        576              448.0         77.8
Professional        452              356.0         78.8


## Evaluate the transparent rule
True positives are non-renewals that the rule identified.

In [3]:
print(pd.read_sql_query("SELECT cohort, clients, flagged, true_positives, false_positives, false_negatives, ROUND(100.0 * true_positives / NULLIF(flagged,0),1) AS precision_pct FROM rule_evaluation", db).to_string(index=False))

  cohort  clients  flagged  true_positives  false_positives  false_negatives  precision_pct
 Holdout      406      116              45               71               49           38.8
Training      794      224              71              153              103           31.7


## Inspect a false positive
A flagged client can still renew. This is not a model defect by itself.

In [4]:
print(pd.read_sql_query("SELECT client_id, recent_activity, prior_activity, expert_sessions, open_tickets, rule_flag, renewed FROM features WHERE cohort='Holdout' AND rule_flag=1 AND renewed=1 LIMIT 5", db).to_string(index=False))
db.close()

client_id  recent_activity  prior_activity  expert_sessions  open_tickets  rule_flag  renewed
    C0029                3               6                0             1          1      1.0
    C0037                1               6                0             0          1      1.0
    C0042                4               9                0             0          1      1.0
    C0059                9              20                0             0          1      1.0
    C0067                0              10                0             0          1      1.0
